# Government of Canada Algorithmic Impact Assessments
## Live publication, impact, version, and data-completeness analysis

This notebook is a **self-contained data-analysis report**. It does not read any previously generated AIA CSVs, Parquet files, JSON caches, or compiler outputs.

Every run starts from the public Open Government Portal CKAN API:

`https://open.canada.ca/data/api/action/package_search?q=collection:aia`

From there it:

1. retrieves and paginates the complete `collection:aia` catalogue inventory;
2. discovers JSON resources attached to each AIA dataset;
3. downloads the published AIA JSON directly from `open.canada.ca`;
4. retrieves the questionnaire definition matching each JSON's embedded AIA version from the official `canada-ca/aia-eia-js` GitHub repository;
5. calculates impact level using the AIA application's scoring approach;
6. measures questionnaire completeness twice:
   - across all questions;
   - across **non-conditional questions only**;
7. analyzes the AIA **Project Phase** (`projectDetailsPhase`) by publication year and department;
8. measures selected phase-aware data-quality, consultation, and data-source answers, including four conditional question pairs;
9. produces twenty-three report figures and three compact analytical CSVs; and
10. writes a standalone HTML report with the plots embedded.

### Scope and interpretation

- Every AIA dataset returned by the CKAN collection search remains in the analysis **except** package `5423054a-093c-4239-85be-fa0b36ae0b2e`, which is documentation rather than an AIA analysis.
- An AIA with no successfully downloadable JSON is assigned **0% completeness** and is explicitly flagged as missing JSON.
- Impact level and AIA version are only available for successfully parsed JSON-backed AIAs.
- Completeness is a **structural population metric**, not a policy-compliance score.
- A question is considered conditional if it has its own SurveyJS `visibleIf` expression **or is nested beneath a page/panel/container with a `visibleIf` expression**.
- The earlier question-specific analyses for system developer, configurator, and system type remain excluded. This report now includes the requested Open Government Portal publication and input-data security-classification questions.

In [ ]:
# @title
from __future__ import annotations

import base64
import html
import json
import math
import os
import re
import sys
import textwrap
from datetime import datetime, timezone
from pathlib import Path
from typing import Any
from urllib.parse import urljoin

import pandas as pd
import matplotlib.pyplot as plt
import requests
from IPython.display import display, Markdown
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

OPEN_CANADA_BASE = "https://open.canada.ca"
PACKAGE_SEARCH_URL = "https://open.canada.ca/data/api/action/package_search"
AIA_COLLECTION_QUERY = "collection:aia"

# This CKAN package is AIA documentation, not an actual assessment.
EXCLUDED_PACKAGE_IDS = {
    "5423054a-093c-4239-85be-fa0b36ae0b2e",
}
EXCLUDED_PACKAGE_REASONS = {
    "5423054a-093c-4239-85be-fa0b36ae0b2e":
        "AIA documentation; not an individual algorithmic impact assessment.",
}
AIA_SURVEY_RAW = (
    "https://raw.githubusercontent.com/"
    "canada-ca/aia-eia-js/{ref}/src/survey-enfr.json"
)

PAGE_SIZE = 100
REQUEST_TIMEOUT = 90

# Output only; nothing is read from this directory.
SAVE_OUTPUTS_TO_GOOGLE_DRIVE = True
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/AIA_analysis_report"
LOCAL_OUTPUT_DIR = os.environ.get("AIA_REPORT_OUTPUT_DIR", "/content/AIA_analysis_report")

if SAVE_OUTPUTS_TO_GOOGLE_DRIVE and "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    OUTPUT_DIR = Path(DRIVE_OUTPUT_DIR)
else:
    OUTPUT_DIR = Path(LOCAL_OUTPUT_DIR)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

print("Output directory:", OUTPUT_DIR)
print("Catalogue source:", PACKAGE_SEARCH_URL + "?q=" + AIA_COLLECTION_QUERY)

## 1. Retrieve the live AIA catalogue inventory

The catalogue is paginated until all records reported by CKAN have been retrieved. Results are then post-filtered to records whose `collection` field is `aia`.

In [ ]:
# @title
def build_session() -> requests.Session:
    session = requests.Session()
    retry = Retry(
        total=5,
        connect=5,
        read=5,
        backoff_factor=1.0,
        status_forcelist=(429, 500, 502, 503, 504),
        allowed_methods=frozenset(["GET"]),
        raise_on_status=False,
    )
    session.mount("https://", HTTPAdapter(max_retries=retry))
    session.headers.update(
        {"User-Agent": "OpenCanada-AIA-Analysis-Report/2.0"}
    )
    return session

SESSION = build_session()

def get_json(url: str, *, params: dict | None = None) -> Any:
    response = SESSION.get(url, params=params, timeout=REQUEST_TIMEOUT)
    response.raise_for_status()
    return response.json()

def package_search_all() -> list[dict]:
    packages = []
    start = 0
    reported_total = None

    while True:
        payload = get_json(
            PACKAGE_SEARCH_URL,
            params={
                "q": AIA_COLLECTION_QUERY,
                "rows": PAGE_SIZE,
                "start": start,
                "sort": "metadata_modified desc",
            },
        )

        if not payload.get("success"):
            raise RuntimeError(f"CKAN package_search failed: {payload}")

        result = payload.get("result") or {}
        batch = result.get("results") or []

        if reported_total is None:
            reported_total = int(result.get("count", len(batch)))

        if not batch:
            break

        packages.extend(batch)
        start += len(batch)

        if start >= reported_total:
            break

    packages = [
        p for p in packages
        if str(p.get("collection", "")).strip().lower() == "aia"
    ]

    return list({p["id"]: p for p in packages if p.get("id")}.values())

packages_search_raw = package_search_all()

excluded_packages_raw = [
    pkg for pkg in packages_search_raw
    if pkg.get("id") in EXCLUDED_PACKAGE_IDS
]

packages_raw = [
    pkg for pkg in packages_search_raw
    if pkg.get("id") not in EXCLUDED_PACKAGE_IDS
]

print(
    f"CKAN returned {len(packages_search_raw):,} collection:aia records. "
    f"Excluded {len(excluded_packages_raw):,} documentation record(s); "
    f"{len(packages_raw):,} AIA analyses remain."
)

if excluded_packages_raw:
    excluded_display_rows = []
    for pkg in excluded_packages_raw:
        title_value = pkg.get("title_translated") or pkg.get("title")
        if isinstance(title_value, dict):
            title_value = title_value.get("en") or title_value.get("default")
        excluded_display_rows.append(
            {
                "package_id": pkg.get("id"),
                "title": title_value,
                "reason": EXCLUDED_PACKAGE_REASONS.get(pkg.get("id")),
            }
        )
    display(pd.DataFrame(excluded_display_rows))

## 2. Normalize packages and discover JSON resources

A dataset can have zero, one, or several JSON resources. The analysis downloads all JSON candidates, then chooses one canonical resource per AIA package, preferring bilingual resources, then English, then French, then other languages.

In [ ]:
# @title
def localized(value: Any) -> tuple[str | None, str | None]:
    if value is None:
        return None, None
    if isinstance(value, str):
        return value, value
    if isinstance(value, dict):
        return (
            value.get("en", value.get("default")),
            value.get("fr", value.get("default", value.get("en"))),
        )
    return str(value), str(value)

def absolute_url(url: str | None) -> str | None:
    return urljoin(OPEN_CANADA_BASE, url) if url else None

package_rows = []
resource_rows = []

for pkg in packages_raw:
    title_en, title_fr = localized(
        pkg.get("title_translated") or pkg.get("title")
    )
    org_at_pub_en, org_at_pub_fr = localized(
        pkg.get("org_title_at_publication")
    )
    org = pkg.get("organization") or {}

    package_rows.append(
        {
            "package_id": pkg.get("id"),
            "package_name": pkg.get("name"),
            "package_title_en": title_en,
            "package_title_fr": title_fr,
            "organization_en": org_at_pub_en or org.get("title"),
            "organization_fr": org_at_pub_fr,
            "organization_slug": org.get("name"),
            "owner_org": pkg.get("owner_org"),
            "date_published": pkg.get("date_published"),
            "portal_release_date": pkg.get("portal_release_date"),
            "metadata_created": pkg.get("metadata_created"),
            "metadata_modified": pkg.get("metadata_modified"),
            "dataset_url": f"{OPEN_CANADA_BASE}/data/en/dataset/{pkg.get('id')}",
        }
    )

    for res in pkg.get("resources") or []:
        name_en, name_fr = localized(
            res.get("name_translated") or res.get("name")
        )
        description_en, description_fr = localized(
            res.get("description_translated") or res.get("description")
        )
        langs = res.get("language") or []
        if isinstance(langs, str):
            langs = [langs]

        url = absolute_url(res.get("url"))
        fmt = str(res.get("format") or "").upper().strip()

        resource_rows.append(
            {
                "package_id": pkg.get("id"),
                "package_title_en": title_en,
                "organization_en": org_at_pub_en or org.get("title"),
                "resource_id": res.get("id"),
                "resource_name_en": name_en,
                "resource_name_fr": name_fr,
                "resource_description_en": description_en,
                "resource_description_fr": description_fr,
                "resource_format": fmt,
                "resource_type": res.get("resource_type"),
                "resource_languages": ",".join(map(str, langs)),
                "resource_position": res.get("position"),
                "resource_last_modified": res.get("last_modified"),
                "resource_url": url,
                "is_json": (
                    fmt == "JSON"
                    or bool(re.search(r"\.json(?:$|[?#])", str(url or ""), flags=re.I))
                ),
            }
        )

packages_df = pd.DataFrame(package_rows)
resources_df = pd.DataFrame(resource_rows)

if resources_df.empty:
    raise RuntimeError("The AIA catalogue records contained no resources.")

json_candidates_df = resources_df[
    resources_df["is_json"].fillna(False)
].copy()

display(
    Markdown(
        f"**Live inventory:** {len(packages_df):,} AIA datasets · "
        f"{len(resources_df):,} resources · "
        f"{len(json_candidates_df):,} JSON resource candidates"
    )
)

## 3. Download the published AIA JSON

Download failures are retained as analytical metadata. A package is considered JSON-backed only if at least one candidate resource parses successfully as a JSON object containing an AIA `data` object.

In [ ]:
# @title
payloads: dict[str, dict] = {}
download_rows = []

for row in json_candidates_df.itertuples(index=False):
    rid = row.resource_id
    status = "ok"
    error = None
    payload = None

    try:
        response = SESSION.get(row.resource_url, timeout=REQUEST_TIMEOUT)
        response.raise_for_status()
        payload = response.json()

        if not isinstance(payload, dict):
            raise ValueError("JSON root is not an object.")
        if not isinstance(payload.get("data"), dict):
            raise ValueError("JSON does not contain an AIA 'data' object.")

        payloads[rid] = payload

    except Exception as exc:
        status = "error"
        error = repr(exc)

    download_rows.append(
        {
            "resource_id": rid,
            "download_status": status,
            "download_error": error,
            "aia_version": payload.get("version") if payload else None,
            "current_page": payload.get("currentPage") if payload else None,
            "data_field_count": len(payload.get("data", {})) if payload else None,
        }
    )

download_df = pd.DataFrame(download_rows)
json_resources_df = json_candidates_df.merge(
    download_df, on="resource_id", how="left"
)

def language_priority(value: str) -> int:
    langs = {
        x.strip().lower()
        for x in str(value or "").split(",")
        if x.strip()
    }
    if {"en", "fr"}.issubset(langs):
        return 0
    if "en" in langs:
        return 1
    if "fr" in langs:
        return 2
    return 3

successful_json_df = json_resources_df[
    json_resources_df["download_status"].eq("ok")
].copy()

successful_json_df["language_priority"] = (
    successful_json_df["resource_languages"].map(language_priority)
)
successful_json_df["_modified"] = pd.to_datetime(
    successful_json_df["resource_last_modified"],
    errors="coerce",
    utc=True,
)

canonical_resources_df = (
    successful_json_df.sort_values(
        ["package_id", "language_priority", "_modified", "resource_position"],
        ascending=[True, True, False, True],
        na_position="last",
    )
    .drop_duplicates("package_id", keep="first")
    .drop(columns=["_modified"])
)

print(
    f"{len(canonical_resources_df):,} of {len(packages_df):,} AIA datasets "
    "have a successfully parsed canonical JSON resource."
)

display(
    json_resources_df[
        [
            "package_title_en",
            "resource_name_en",
            "resource_languages",
            "aia_version",
            "download_status",
            "download_error",
        ]
    ].sort_values(["package_title_en", "resource_name_en"])
)

## 4. Retrieve the matching AIA questionnaire schemas

The embedded `version` in each published AIA JSON is used as the first Git reference attempted against the official AIA repository. If that exact historical ref is unavailable, the notebook falls back to `master` and records the fallback reference.

The schema walker propagates conditional status from parent pages and panels, so a question nested inside a conditional container is treated as conditional even when the question itself has no `visibleIf`.

In [ ]:
# @title
_SURVEY_CACHE: dict[str, tuple[dict, str]] = {}

def load_survey_for_version(version: str | None) -> tuple[dict, str]:
    version = str(version or "").strip()
    key = version or "master"

    if key in _SURVEY_CACHE:
        return _SURVEY_CACHE[key]

    refs = []
    if version:
        refs.append(version)
        if not version.startswith("v"):
            refs.append("v" + version)
    refs.append("master")

    last_error = None

    for ref in dict.fromkeys(refs):
        try:
            survey = get_json(AIA_SURVEY_RAW.format(ref=ref))
            if not isinstance(survey, dict):
                raise ValueError("Survey definition is not a JSON object.")
            _SURVEY_CACHE[key] = (survey, ref)
            return survey, ref
        except Exception as exc:
            last_error = exc

    raise RuntimeError(
        f"Unable to load questionnaire schema for {version!r}. "
        f"Last error: {last_error}"
    )

def _localized_text_en(value: Any) -> str:
    """Return stable English/default display text from a SurveyJS value."""
    if isinstance(value, dict):
        value = value.get("default") or value.get("en") or value.get("fr") or ""
    return re.sub(r"\s+", " ", html.unescape(str(value or ""))).strip()

def iter_schema_questions(
    node: Any,
    *,
    parent_name: str | None = None,
    inherited_conditional: bool = False,
):
    if isinstance(node, list):
        for item in node:
            yield from iter_schema_questions(
                item,
                parent_name=parent_name,
                inherited_conditional=inherited_conditional,
            )
        return

    if not isinstance(node, dict):
        return

    node_name = node.get("name")
    node_type = str(node.get("type") or "")
    own_conditional = bool(str(node.get("visibleIf") or "").strip())
    effective_conditional = inherited_conditional or own_conditional

    non_question_types = {"", "panel", "html", "expression", "image"}

    if node_name and node_type not in non_question_types:
        yield {
            "question_id": str(node_name),
            "question_type": node_type,
            "question_text_en": _localized_text_en(node.get("title")),
            "parent_name": parent_name,
            "visible_if": node.get("visibleIf"),
            "is_conditional": effective_conditional,
            "is_required": bool(node.get("isRequired", False)),
            "choices": node.get("choices") or [],
        }

    child_parent = node_name or parent_name

    for key in ("pages", "elements", "templateElements"):
        if key in node:
            yield from iter_schema_questions(
                node[key],
                parent_name=child_parent,
                inherited_conditional=effective_conditional,
            )

def build_schema_spec(
    survey: dict,
    *,
    version: str | None,
    schema_ref: str,
) -> pd.DataFrame:
    rows = list(iter_schema_questions(survey))
    df = pd.DataFrame(rows)

    if df.empty:
        return pd.DataFrame(
            columns=[
                "aia_version",
                "schema_ref",
                "question_id",
                "question_type",
                "question_text_en",
                "parent_name",
                "visible_if",
                "is_conditional",
                "is_required",
                "choices",
            ]
        )

    df.insert(0, "schema_ref", schema_ref)
    df.insert(0, "aia_version", version)
    return df.drop_duplicates("question_id", keep="last")

schema_specs: dict[str, pd.DataFrame] = {}
schema_refs: dict[str, str] = {}

observed_versions = sorted(
    {
        str(v)
        for v in canonical_resources_df["aia_version"].dropna().unique()
        if str(v).strip()
    }
)

for version in observed_versions:
    survey, ref = load_survey_for_version(version)
    schema_specs[version] = build_schema_spec(
        survey,
        version=version,
        schema_ref=ref,
    )
    schema_refs[version] = ref

schema_inventory = pd.DataFrame(
    [
        {
            "aia_version": version,
            "schema_ref": schema_refs[version],
            "question_count": len(schema_specs[version]),
            "nonconditional_question_count": int(
                (~schema_specs[version]["is_conditional"]).sum()
            ),
        }
        for version in observed_versions
    ]
)

# The current master survey supplies display wording when historical versions
# vary. Historical schemas still control answer decoding and field resolution.
latest_survey, latest_schema_ref = load_survey_for_version("master")
latest_schema_spec = build_schema_spec(
    latest_survey,
    version="master",
    schema_ref=latest_schema_ref,
)

display(schema_inventory)

## 5. Calculate AIA impact levels

The score calculation mirrors the AIA application's encoded-answer scoring convention. The questionnaire version determines both the score-bearing questions and the maximum possible raw-risk score.

In [ ]:
# @title
def embedded_score(value: Any) -> float:
    if value is None or isinstance(value, bool):
        return 0.0
    if isinstance(value, list):
        return float(sum(embedded_score(v) for v in value))
    if isinstance(value, (int, float)):
        return float(value)
    if isinstance(value, str):
        tail = value.rsplit("-", 1)[-1] if "-" in value else ""
        try:
            return float(tail)
        except (TypeError, ValueError):
            return 0.0
    return 0.0

def score_type(name: str | None, parent_name: str | None) -> str | None:
    for candidate in (name, parent_name):
        candidate = str(candidate or "")
        if candidate.endswith("-RS"):
            return "raw_risk"
        if candidate.endswith("-MS"):
            return "mitigation"
        if candidate.endswith("-NS"):
            return "not_scored"
    return None

def question_max_score(question_type: str, choices: Any) -> float:
    values = []
    for choice in choices or []:
        value = choice.get("value") if isinstance(choice, dict) else choice
        values.append(embedded_score(value))

    if question_type == "checkbox":
        return float(sum(values))
    if question_type in {"radiogroup", "dropdown"}:
        return float(max([0.0] + values))
    return 0.0

def calculate_impact(payload: dict) -> dict:
    version = str(payload.get("version") or "").strip()
    survey, schema_ref = load_survey_for_version(version)

    raw = 0.0
    raw_max = 0.0
    mitigation = 0.0
    mitigation_max = 0.0
    data = payload.get("data") or {}

    for q in iter_schema_questions(survey):
        qtype = score_type(q["question_id"], q["parent_name"])
        if qtype not in {"raw_risk", "mitigation"}:
            continue

        observed = embedded_score(data.get(q["question_id"]))
        max_score = question_max_score(q["question_type"], q["choices"])

        if qtype == "raw_risk":
            raw += observed
            raw_max += max_score
        else:
            mitigation += observed
            mitigation_max += max_score

    mitigation_threshold = 0.80 * (mitigation_max / 2)

    if mitigation >= mitigation_threshold:
        final_score = int(math.floor((0.85 * raw) + 0.5))
        deduction_applied = True
    else:
        final_score = int(math.floor(raw + 0.5))
        deduction_applied = False

    if raw_max <= 0:
        level = None
    elif final_score <= raw_max * 0.25:
        level = 1
    elif final_score <= raw_max * 0.50:
        level = 2
    elif final_score <= raw_max * 0.75:
        level = 3
    else:
        level = 4

    return {
        "aia_version": version or None,
        "score_schema_ref": schema_ref,
        "raw_risk_score": raw,
        "max_raw_risk_score": raw_max,
        "mitigation_score": mitigation,
        "max_mitigation_score": mitigation_max,
        "mitigation_deduction_applied": deduction_applied,
        "final_score": final_score,
        "impact_level": level,
    }

score_rows = []

for row in canonical_resources_df.itertuples(index=False):
    payload = payloads.get(row.resource_id)

    try:
        result = calculate_impact(payload)
        result["resource_id"] = row.resource_id
        result["scoring_error"] = None
    except Exception as exc:
        result = {
            "resource_id": row.resource_id,
            "aia_version": payload.get("version") if isinstance(payload, dict) else None,
            "score_schema_ref": None,
            "raw_risk_score": None,
            "max_raw_risk_score": None,
            "mitigation_score": None,
            "max_mitigation_score": None,
            "mitigation_deduction_applied": None,
            "final_score": None,
            "impact_level": None,
            "scoring_error": repr(exc),
        }

    score_rows.append(result)

scores_df = pd.DataFrame(score_rows)

display(
    scores_df[
        [
            "aia_version",
            "score_schema_ref",
            "impact_level",
            "final_score",
            "max_raw_risk_score",
            "scoring_error",
        ]
    ]
)

## 6. Build one analytical row per published AIA

For each AIA with JSON, completeness uses the questionnaire definition associated with that JSON's embedded version. AIAs without usable JSON receive 0% for both completeness measures.

In [ ]:
# @title
def has_substantive_answer(value: Any) -> bool:
    if value is None:
        return False

    if isinstance(value, (list, tuple, set)):
        return any(has_substantive_answer(v) for v in value)

    if isinstance(value, dict):
        return any(has_substantive_answer(v) for v in value.values())

    if isinstance(value, str):
        return value.strip().lower() not in {"", "nan", "none", "null"}

    try:
        return not bool(pd.isna(value))
    except (TypeError, ValueError):
        return True

publication_date = pd.to_datetime(
    packages_df["date_published"], errors="coerce", utc=True
)
publication_date = publication_date.fillna(
    pd.to_datetime(packages_df["portal_release_date"], errors="coerce", utc=True)
)
publication_date = publication_date.fillna(
    pd.to_datetime(packages_df["metadata_created"], errors="coerce", utc=True)
)

report = packages_df.copy()
report["publication_date"] = publication_date
report["publication_year"] = publication_date.dt.year.astype("Int64")

canonical_meta = canonical_resources_df[
    [
        "package_id",
        "resource_id",
        "resource_url",
        "resource_languages",
        "aia_version",
        "download_status",
    ]
].copy()

report = report.merge(canonical_meta, on="package_id", how="left")

report["has_usable_json"] = (
    report["resource_id"].notna()
    & report["download_status"].fillna("").eq("ok")
)
report["missing_json"] = ~report["has_usable_json"]

report = report.merge(
    scores_df.drop(columns=["aia_version"], errors="ignore"),
    on="resource_id",
    how="left",
)

PROJECT_PHASE_QUESTION_ID = "projectDetailsPhase"

def _choice_label_en(choice: Any) -> tuple[str, str]:
    """Return (choice_value, English label) from a SurveyJS choice."""
    if isinstance(choice, dict):
        value = str(choice.get("value"))
        text = choice.get("text")
        if isinstance(text, dict):
            label = (
                text.get("default")
                or text.get("en")
                or text.get("fr")
                or value
            )
        elif text is None:
            label = value
        else:
            label = str(text)
        return value, str(label)

    value = str(choice)
    return value, value

def question_choice_map(
    version: str | None,
    question_id: str,
) -> dict[str, str]:
    """Decode choice values using the questionnaire matching the AIA version."""
    schema = schema_specs.get(str(version or "").strip())
    if schema is None or schema.empty:
        return {}

    matches = schema[
        schema["question_id"].astype(str).eq(question_id)
    ]
    if matches.empty:
        return {}

    choices = matches.iloc[-1]["choices"]
    if not isinstance(choices, list):
        return {}

    return dict(_choice_label_en(choice) for choice in choices)

# Build a stable cross-version display dictionary. Raw codes remain the
# grouping key, while labels come from the version-specific schemas.
PROJECT_PHASE_LABEL_BY_CODE: dict[str, str] = {}

for version in observed_versions:
    for code, label in question_choice_map(
        version,
        PROJECT_PHASE_QUESTION_ID,
    ).items():
        PROJECT_PHASE_LABEL_BY_CODE.setdefault(code, label)

project_phase_rows = []

for row in report.itertuples(index=False):
    code = None
    label = None

    if bool(row.has_usable_json):
        payload = payloads.get(row.resource_id) or {}
        data = payload.get("data") or {}
        raw_phase = data.get(PROJECT_PHASE_QUESTION_ID)

        if has_substantive_answer(raw_phase):
            code = str(raw_phase)
            version_map = question_choice_map(
                payload.get("version"),
                PROJECT_PHASE_QUESTION_ID,
            )
            label = (
                version_map.get(code)
                or PROJECT_PHASE_LABEL_BY_CODE.get(code)
                or code
            )

    project_phase_rows.append(
        {
            "package_id": row.package_id,
            "project_phase_code": code,
            "project_phase_label": label,
        }
    )

report = report.merge(
    pd.DataFrame(project_phase_rows),
    on="package_id",
    how="left",
)


# -------------------------------------------------------------------
# Selected phase-aware data-quality, consultation, and data-source answers
# -------------------------------------------------------------------
SELECTED_QUESTION_LABELS = {
    "dataQualityPhase1": "Documented bias-testing process",
    "dataQualityPhase2": "Bias-testing process publicly available",
    "dataQualityPhase3": "Data-quality resolution process",
    "dataQualityPhase4": "Resolution process publicly available",
    "dataQualityPhase5": "Gender Based Analysis Plus of the data",
    "dataQualityPhaseGbaPublic": "GBA Plus findings publicly available",
    "dataQualityPhase8": "Unreliable-data risk process",
    "dataQualityPhase9": "Risk process publicly available",
    "dataQualityPhase10": "Data posted on the Open Government Portal",
    "consultationPhase1": "Internal consultees or partners",
    "consultationPhase3": "External consultees or partners",
    "aboutDataSource2": "Highest input-data security classification",
}

PHASE_QUESTION_SOURCES = {
    "dataQualityPhase1": ("dataQuality", ("1",)),
    "dataQualityPhase2": ("dataQuality", ("2",)),
    "dataQualityPhase3": ("dataQuality", ("3",)),
    "dataQualityPhase4": ("dataQuality", ("4",)),
    "dataQualityPhase5": ("dataQuality", ("5",)),
    # In v1+, question 6 is a Describe field and public availability moved to
    # question 7. In v0.x surveys the public-availability field is question 6.
    "dataQualityPhaseGbaPublic": ("dataQuality", ("7", "6")),
    "dataQualityPhase8": ("dataQuality", ("8",)),
    "dataQualityPhase9": ("dataQuality", ("9",)),
    "dataQualityPhase10": ("dataQuality", ("10",)),
    "consultationPhase1": ("consultation", ("1",)),
    "consultationPhase3": ("consultation", ("3",)),
}

PAIRED_QUESTION_GROUPS = [
    (
        "dataQualityPhase1",
        "dataQualityPhase2",
        "Bias testing and public availability",
    ),
    (
        "dataQualityPhase3",
        "dataQualityPhase4",
        "Data-quality resolution and public availability",
    ),
    (
        "dataQualityPhase8",
        "dataQualityPhase9",
        "Unreliable-data risk management and public availability",
    ),
    (
        "dataQualityPhase5",
        "dataQualityPhaseGbaPublic",
        "GBA Plus analysis and public availability",
    ),
]

PAIR_PLOT_LABELS = {
    "dataQualityPhase1": "Bias-testing process documented",
    "dataQualityPhase2": "Process publicly available",
    "dataQualityPhase3": "Resolution process documented",
    "dataQualityPhase4": "Process publicly available",
    "dataQualityPhase5": "GBA Plus analysis undertaken",
    "dataQualityPhaseGbaPublic": "Findings publicly available",
    "dataQualityPhase8": "Risk-management process documented",
    "dataQualityPhase9": "Process publicly available",
}

STANDALONE_QUESTION_IDS = [
    "dataQualityPhase10",
    "aboutDataSource2",
]

CONSULTATION_QUESTION_IDS = [
    "consultationPhase1",
    "consultationPhase3",
]

SELECTED_QUESTION_IDS = list(SELECTED_QUESTION_LABELS)
PAIR_SECOND_TO_FIRST = {
    second: first
    for first, second, _ in PAIRED_QUESTION_GROUPS
}

LATEST_LOGICAL_SOURCE_IDS = {
    "dataQualityPhase1": ("dataQualityDesign1", "dataQualityImplementation1"),
    "dataQualityPhase2": ("dataQualityDesign2", "dataQualityImplementation2"),
    "dataQualityPhase3": ("dataQualityDesign3", "dataQualityImplementation3"),
    "dataQualityPhase4": ("dataQualityDesign4", "dataQualityImplementation4"),
    "dataQualityPhase5": ("dataQualityDesign5", "dataQualityImplementation5"),
    "dataQualityPhaseGbaPublic": (
        "dataQualityDesign7",
        "dataQualityImplementation7",
    ),
    "dataQualityPhase8": ("dataQualityDesign8", "dataQualityImplementation8"),
    "dataQualityPhase9": ("dataQualityDesign9", "dataQualityImplementation9"),
    "dataQualityPhase10": (
        "dataQualityDesign10",
        "dataQualityImplementation10",
    ),
}

def schema_question_text(schema: pd.DataFrame, question_id: str) -> str:
    matches = schema[schema["question_id"].astype(str).eq(question_id)]
    if matches.empty:
        return question_id
    return str(matches.iloc[-1]["question_text_en"] or question_id)

def latest_data_quality_wording(logical_question_id: str) -> str:
    """Return full Design and Implementation wording from the latest survey."""
    design_id, implementation_id = LATEST_LOGICAL_SOURCE_IDS[logical_question_id]
    design_text = schema_question_text(latest_schema_spec, design_id)
    implementation_text = schema_question_text(latest_schema_spec, implementation_id)
    if design_text.strip().casefold() == implementation_text.strip().casefold():
        return f"Latest survey wording: {design_text}"
    return (
        f"Latest survey wording — Design: {design_text}\n"
        f"Implementation: {implementation_text}"
    )

DATA_QUALITY_FULL_QUESTION_TEXT = {
    question_id: latest_data_quality_wording(question_id)
    for question_id in LATEST_LOGICAL_SOURCE_IDS
}

def normalize_answer_label(value: Any) -> str | None:
    if not has_substantive_answer(value):
        return None

    label = str(value).strip()
    normalized = label.casefold()
    if normalized in {"yes", "oui"}:
        return "Yes"
    if normalized in {"no", "non"}:
        return "No"
    return label

def project_phase_variant(project_phase_label: Any) -> str | None:
    """Return the questionnaire field infix for the saved project phase."""
    normalized = str(project_phase_label or "").strip().casefold()
    if normalized == "design":
        return "Design"
    if normalized == "implementation":
        return "Implementation"
    return None

def ordered_source_suffixes(
    logical_question_id: str,
    version: str,
) -> tuple[str, ...]:
    prefix, suffixes = PHASE_QUESTION_SOURCES[logical_question_id]
    if logical_question_id != "dataQualityPhaseGbaPublic":
        return suffixes
    normalized_version = str(version or "").strip().lower().lstrip("v")
    return ("7", "6") if normalized_version.startswith("1.") else ("6", "7")

def resolve_source_question_id(
    logical_question_id: str,
    project_phase_label: Any,
    version: str,
    data: dict,
    schema_question_ids: set[str],
) -> str:
    """Resolve a logical phase question to its saved survey field."""
    source_spec = PHASE_QUESTION_SOURCES.get(logical_question_id)
    if source_spec is None:
        return logical_question_id

    prefix, _ = source_spec
    suffixes = ordered_source_suffixes(logical_question_id, version)
    phase = project_phase_variant(project_phase_label)
    phases = [phase] if phase else []
    phases.extend(candidate for candidate in ("Design", "Implementation") if candidate not in phases)
    candidates = [
        f"{prefix}{candidate_phase}{suffix}"
        for candidate_phase in phases
        for suffix in suffixes
    ]

    # Project Phase and version determine the preferred source. Explicit saved
    # answers and schema presence provide fallbacks for malformed older files.
    preferred = [
        f"{prefix}{phase}{suffix}"
        for suffix in suffixes
    ] if phase else []
    for question_id in preferred:
        if has_substantive_answer(data.get(question_id)):
            return question_id
    for question_id in preferred:
        if question_id in schema_question_ids:
            return question_id
    for question_id in candidates:
        if has_substantive_answer(data.get(question_id)):
            return question_id
    for question_id in candidates:
        if question_id in schema_question_ids:
            return question_id
    return candidates[0]

selected_answer_rows = []

for row in report.itertuples(index=False):
    if not bool(row.has_usable_json):
        continue

    payload = payloads.get(row.resource_id) or {}
    data = payload.get("data") or {}
    version = str(payload.get("version") or "").strip()
    schema = schema_specs.get(version)
    schema_question_ids = (
        set(schema["question_id"].astype(str))
        if schema is not None and not schema.empty
        else set()
    )

    decoded_answers = {}
    for question_id in SELECTED_QUESTION_IDS:
        source_question_id = resolve_source_question_id(
            question_id,
            row.project_phase_label,
            version,
            data,
            schema_question_ids,
        )
        raw_answer = data.get(source_question_id)
        answer_code = str(raw_answer) if has_substantive_answer(raw_answer) else None
        choice_map = question_choice_map(version, source_question_id)
        answer_label = normalize_answer_label(
            choice_map.get(answer_code, answer_code)
            if answer_code is not None
            else None
        )
        decoded_answers[question_id] = {
            "source_question_id": source_question_id,
            "answer_code": answer_code,
            "answer_label": answer_label,
        }

    for question_id in SELECTED_QUESTION_IDS:
        decoded = decoded_answers[question_id]
        source_question_id = decoded["source_question_id"]
        question_present = source_question_id in schema_question_ids
        applicable = question_present

        first_question = PAIR_SECOND_TO_FIRST.get(question_id)
        if first_question:
            applicable = question_present and (
                decoded_answers[first_question]["answer_label"] == "Yes"
                or decoded["answer_label"] is not None
            )

        answered = applicable and decoded["answer_label"] is not None

        selected_answer_rows.append(
            {
                "package_id": row.package_id,
                "package_title_en": row.package_title_en,
                "organization_en": row.organization_en,
                "owner_org": row.owner_org,
                "publication_year": row.publication_year,
                "aia_version": version,
                "project_phase_label": row.project_phase_label,
                "question_id": question_id,
                "source_question_id": source_question_id,
                "question_label": SELECTED_QUESTION_LABELS[question_id],
                "applicable": applicable,
                "answered": answered,
                "answer_code": decoded["answer_code"] if answered else None,
                "answer_label": decoded["answer_label"] if answered else None,
            }
        )

question_answers_df = pd.DataFrame(selected_answer_rows)
SELECTED_QUESTION_OUTPUT_COLUMNS = []

for question_id in SELECTED_QUESTION_IDS:
    output_columns = {
        "source_question_id": f"{question_id}_source_question_id",
        "applicable": f"{question_id}_applicable",
        "answered": f"{question_id}_answered",
        "answer_code": f"{question_id}_answer_code",
        "answer_label": f"{question_id}_answer_label",
    }
    SELECTED_QUESTION_OUTPUT_COLUMNS.extend(output_columns.values())

    question_wide = question_answers_df.loc[
        question_answers_df["question_id"].eq(question_id),
        ["package_id", *output_columns],
    ].rename(columns=output_columns)

    report = report.merge(question_wide, on="package_id", how="left")
    for boolean_column in (
        output_columns["applicable"],
        output_columns["answered"],
    ):
        report[boolean_column] = report[boolean_column].eq(True)


completion_rows = []

for row in report.itertuples(index=False):
    if not bool(row.has_usable_json):
        completion_rows.append(
            {
                "package_id": row.package_id,
                "expected_all_questions": 0,
                "answered_all_questions": 0,
                "completion_all_pct": 0.0,
                "expected_nonconditional_questions": 0,
                "answered_nonconditional_questions": 0,
                "completion_nonconditional_pct": 0.0,
                "completeness_note": (
                    "Missing or unusable JSON; both completeness "
                    "measures scored as 0%."
                ),
            }
        )
        continue

    payload = payloads.get(row.resource_id)
    version = str(payload.get("version") or "").strip()
    schema = schema_specs.get(version)

    if schema is None or schema.empty:
        completion_rows.append(
            {
                "package_id": row.package_id,
                "expected_all_questions": 0,
                "answered_all_questions": 0,
                "completion_all_pct": 0.0,
                "expected_nonconditional_questions": 0,
                "answered_nonconditional_questions": 0,
                "completion_nonconditional_pct": 0.0,
                "completeness_note": (
                    f"No questionnaire schema was resolved for "
                    f"{version}; scored as 0%."
                ),
            }
        )
        continue

    data = payload.get("data") or {}

    all_ids = set(schema["question_id"].astype(str))
    nonconditional_ids = set(
        schema.loc[~schema["is_conditional"], "question_id"].astype(str)
    )

    answered_ids = {
        str(k)
        for k, value in data.items()
        if has_substantive_answer(value)
    }

    answered_all = len(all_ids & answered_ids)
    answered_nonconditional = len(nonconditional_ids & answered_ids)

    completion_rows.append(
        {
            "package_id": row.package_id,
            "expected_all_questions": len(all_ids),
            "answered_all_questions": answered_all,
            "completion_all_pct": (
                round(100 * answered_all / len(all_ids), 2)
                if all_ids else 0.0
            ),
            "expected_nonconditional_questions": len(nonconditional_ids),
            "answered_nonconditional_questions": answered_nonconditional,
            "completion_nonconditional_pct": (
                round(
                    100 * answered_nonconditional / len(nonconditional_ids),
                    2,
                )
                if nonconditional_ids else 0.0
            ),
            "completeness_note": (
                "Calculated from the live JSON and matching "
                "version-specific questionnaire schema."
            ),
        }
    )

report = report.merge(
    pd.DataFrame(completion_rows),
    on="package_id",
    how="left",
)

report["impact_level"] = pd.to_numeric(
    report["impact_level"], errors="coerce"
).astype("Int64")

assessment_report = report[
    [
        "package_id",
        "package_title_en",
        "organization_en",
        "owner_org",
        "dataset_url",
        "publication_date",
        "publication_year",
        "has_usable_json",
        "missing_json",
        "resource_id",
        "resource_url",
        "resource_languages",
        "aia_version",
        "project_phase_code",
        "project_phase_label",
        *SELECTED_QUESTION_OUTPUT_COLUMNS,
        "score_schema_ref",
        "impact_level",
        "final_score",
        "expected_all_questions",
        "answered_all_questions",
        "completion_all_pct",
        "expected_nonconditional_questions",
        "answered_nonconditional_questions",
        "completion_nonconditional_pct",
        "completeness_note",
    ]
].copy()

assessment_report.sort_values(
    ["organization_en", "publication_date", "package_title_en"],
    inplace=True,
    na_position="last",
)

# -------------------------------------------------------------------
# Public peer-review resource coverage
# -------------------------------------------------------------------
#
# Count only a separately published resource that explicitly identifies
# itself as a peer review. A package-description statement that a peer
# review happened does not count unless a matching resource is present.

PEER_REVIEW_EN_RE = re.compile(
    r"\bpeer[\s-]*review\b",
    flags=re.I,
)

PEER_REVIEW_FR_RE = re.compile(
    r"\b(?:examen|évaluation|evaluation|revue)\s+par\s+les\s+pairs\b",
    flags=re.I,
)

def is_peer_review_resource(row) -> bool:
    searchable = " ".join(
        str(getattr(row, field, "") or "")
        for field in (
            "resource_name_en",
            "resource_name_fr",
            "resource_description_en",
            "resource_description_fr",
            "resource_url",
        )
    )
    return bool(
        PEER_REVIEW_EN_RE.search(searchable)
        or PEER_REVIEW_FR_RE.search(searchable)
    )

peer_resources_df = resources_df.copy()
peer_resources_df["is_peer_review_resource"] = [
    is_peer_review_resource(row)
    for row in peer_resources_df.itertuples(index=False)
]
peer_resources_df = peer_resources_df[
    peer_resources_df["is_peer_review_resource"]
].copy()

if peer_resources_df.empty:
    peer_by_package = pd.DataFrame(
        columns=[
            "package_id",
            "peer_review_resource_count",
            "peer_review_resource_ids",
            "peer_review_resource_names",
            "peer_review_resource_urls",
        ]
    )
else:
    peer_by_package = (
        peer_resources_df.groupby("package_id", dropna=False)
        .agg(
            peer_review_resource_count=("resource_id", "nunique"),
            peer_review_resource_ids=(
                "resource_id",
                lambda s: " | ".join(
                    sorted({str(v) for v in s.dropna()})
                ),
            ),
            peer_review_resource_names=(
                "resource_name_en",
                lambda s: " | ".join(
                    dict.fromkeys(
                        str(v)
                        for v in s.dropna()
                        if str(v).strip()
                    )
                ),
            ),
            peer_review_resource_urls=(
                "resource_url",
                lambda s: " | ".join(
                    dict.fromkeys(
                        str(v)
                        for v in s.dropna()
                        if str(v).strip()
                    )
                ),
            ),
        )
        .reset_index()
    )

assessment_report = assessment_report.merge(
    peer_by_package,
    on="package_id",
    how="left",
)

assessment_report["peer_review_resource_count"] = (
    assessment_report["peer_review_resource_count"]
    .fillna(0)
    .astype(int)
)

for column in (
    "peer_review_resource_ids",
    "peer_review_resource_names",
    "peer_review_resource_urls",
):
    assessment_report[column] = assessment_report[column].fillna("")

assessment_report["has_peer_review_resource"] = (
    assessment_report["peer_review_resource_count"] > 0
)

assessment_report["peer_review_level_2_plus_scope"] = (
    assessment_report["impact_level"].fillna(0) >= 2
)

assessment_report["peer_review_found_in_scope"] = (
    assessment_report["peer_review_level_2_plus_scope"]
    & assessment_report["has_peer_review_resource"]
)

assessment_report["peer_review_missing_in_scope"] = (
    assessment_report["peer_review_level_2_plus_scope"]
    & ~assessment_report["has_peer_review_resource"]
)

assessment_report["peer_review_status"] = "Not in level 2+ scope"
assessment_report.loc[
    assessment_report["peer_review_found_in_scope"],
    "peer_review_status",
] = "Peer review resource found"
assessment_report.loc[
    assessment_report["peer_review_missing_in_scope"],
    "peer_review_status",
] = "No peer review resource found"

assessment_report.sort_values(
    ["organization_en", "publication_date", "package_title_en"],
    inplace=True,
    na_position="last",
)

display(assessment_report.head(10))

# Executive summary

In [ ]:
# @title
total = len(assessment_report)
json_count = int(assessment_report["has_usable_json"].sum())
missing_count = int(assessment_report["missing_json"].sum())
institution_count = assessment_report["organization_en"].nunique()
years = assessment_report["publication_year"].dropna()

overall_all = assessment_report["completion_all_pct"].mean()
overall_nonconditional = assessment_report["completion_nonconditional_pct"].mean()

peer_scope_count = int(
    assessment_report["peer_review_level_2_plus_scope"].sum()
)
peer_found_count = int(
    assessment_report["peer_review_found_in_scope"].sum()
)
peer_missing_count = int(
    assessment_report["peer_review_missing_in_scope"].sum()
)
peer_coverage_pct = (
    100 * peer_found_count / peer_scope_count
    if peer_scope_count
    else float("nan")
)

impact_counts = (
    assessment_report["impact_level"]
    .dropna()
    .astype(int)
    .value_counts()
    .sort_index()
)

version_counts = (
    assessment_report.loc[
        assessment_report["has_usable_json"],
        "aia_version",
    ]
    .fillna("Unknown")
    .value_counts()
)

summary_markdown = (
    "### Live snapshot\n\n"
    f"- **{total:,}** AIA dataset records across **{institution_count:,} institutions**.\n"
    f"- **{json_count:,}** have usable published JSON; **{missing_count:,}** do not and receive 0% completeness.\n"
    + (
        f"- Publication years represented: **{int(years.min())}-{int(years.max())}**.\n"
        if not years.empty else ""
    )
    + f"- Mean all-question completeness: **{overall_all:.1f}%**.\n"
    + f"- Mean non-conditional completeness: **{overall_nonconditional:.1f}%**.\n"
    + (
        f"- Peer-review resources for impact level 2+ AIAs: "
        f"**{peer_found_count:,} of {peer_scope_count:,} "
        f"({peer_coverage_pct:.1f}%)**; "
        f"**{peer_missing_count:,}** have no peer-review resource detected.\n"
        if peer_scope_count else ""
    )
    + "- Impact levels: **"
    + ", ".join(f"Level {level}: {count}" for level, count in impact_counts.items())
    + "**.\n"
    + "- AIA versions: **"
    + ", ".join(f"{version}: {count}" for version, count in version_counts.items())
    + "**."
)

display(Markdown(summary_markdown))

# Publication and machine-readable coverage

The first figures use the full catalogue inventory, including AIA records without JSON.

In [ ]:
# @title
PLOT_FILES: list[tuple[str, Path, str | None]] = []

def save_and_show(
    fig,
    filename: str,
    title: str,
    question_text: str | None = None,
):
    path = OUTPUT_DIR / filename
    fig.savefig(path, dpi=180, bbox_inches="tight")
    PLOT_FILES.append((title, path, question_text))
    plt.show()
    return path

annual_publications = (
    assessment_report.dropna(subset=["publication_year"])
    .groupby("publication_year")
    .size()
    .sort_index()
)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(
    annual_publications.index.astype(str),
    annual_publications.values,
)
ax.set_title("Published AIA records by year")
ax.set_xlabel("Publication year")
ax.set_ylabel("AIA records")
ax.bar_label(bars, padding=3)
fig.tight_layout()

save_and_show(
    fig,
    "01_aia_publications_by_year.png",
    "Published AIA records by year",
)

In [ ]:
# @title
institution_json = (
    assessment_report.groupby("organization_en", dropna=False)
    .agg(
        usable_json=("has_usable_json", "sum"),
        missing_json=("missing_json", "sum"),
        total_aias=("package_id", "count"),
    )
    .sort_values(["total_aias", "usable_json"], ascending=True)
)

ax = institution_json[["usable_json", "missing_json"]].plot(
    kind="barh",
    stacked=True,
    figsize=(11, max(5, 0.52 * len(institution_json))),
)
ax.set_title("AIA records by institution and JSON availability")
ax.set_xlabel("AIA records")
ax.set_ylabel("")
ax.legend(["Usable JSON", "Missing / unusable JSON"], loc="lower right")

for container in ax.containers:
    labels = [f"{int(v)}" if v > 0 else "" for v in container.datavalues]
    ax.bar_label(container, labels=labels, label_type="center")

fig = ax.figure
fig.tight_layout()

save_and_show(
    fig,
    "02_aia_json_availability_by_institution.png",
    "AIA records by institution and JSON availability",
)

# Impact level and AIA version

These figures use successfully parsed JSON-backed AIAs because impact level and embedded questionnaire version cannot be reproducibly derived from PDF-only records.

In [ ]:
# @title
impact_data = assessment_report[
    assessment_report["publication_year"].notna()
    & assessment_report["impact_level"].notna()
].copy()

impact_data["impact_label"] = (
    impact_data["impact_level"]
    .astype(int)
    .map(lambda x: f"Impact level {x}")
)

impact_year = pd.crosstab(
    impact_data["publication_year"],
    impact_data["impact_label"],
).sort_index()

ax = impact_year.plot(kind="bar", stacked=True, figsize=(10, 5))
ax.set_title("JSON-backed AIAs by impact level and publication year")
ax.set_xlabel("Publication year")
ax.set_ylabel("AIA count")
ax.tick_params(axis="x", rotation=0)

for container in ax.containers:
    labels = [str(int(v)) if v > 0 else "" for v in container.datavalues]
    ax.bar_label(container, labels=labels, label_type="center")

fig = ax.figure
fig.tight_layout()

save_and_show(
    fig,
    "03_aia_impact_level_by_year.png",
    "JSON-backed AIAs by impact level and publication year",
)

In [ ]:
# @title
version_data = assessment_report[
    assessment_report["has_usable_json"]
    & assessment_report["publication_year"].notna()
].copy()

version_year = pd.crosstab(
    version_data["publication_year"],
    version_data["aia_version"].fillna("Unknown"),
).sort_index()

ax = version_year.plot(kind="bar", stacked=True, figsize=(10, 5))
ax.set_title("JSON-backed AIAs by questionnaire version and publication year")
ax.set_xlabel("Publication year")
ax.set_ylabel("AIA count")
ax.tick_params(axis="x", rotation=0)

for container in ax.containers:
    labels = [str(int(v)) if v > 0 else "" for v in container.datavalues]
    ax.bar_label(container, labels=labels, label_type="center")

fig = ax.figure
fig.tight_layout()

save_and_show(
    fig,
    "04_aia_version_by_year.png",
    "JSON-backed AIAs by questionnaire version and publication year",
)

# Project Phase

`projectDetailsPhase` is decoded from the questionnaire definition matching each AIA's embedded version rather than from a hard-coded lookup. The raw value (for example `item2`) remains in the AIA-level CSV, while the plots use the human-readable phase label.

These plots include **JSON-backed AIAs with an answered Project Phase**. AIAs without JSON cannot be assigned a phase and are not silently classified into a project stage.

In [ ]:
# @title
project_phase_data = assessment_report[
    assessment_report["has_usable_json"]
    & assessment_report["project_phase_code"].notna()
].copy()

if project_phase_data.empty:
    raise ValueError(
        "No JSON-backed AIA contained an answered projectDetailsPhase field."
    )

# Use raw phase code as the stable grouping key, then display the decoded label.
phase_display_by_code = (
    project_phase_data[
        ["project_phase_code", "project_phase_label"]
    ]
    .dropna(subset=["project_phase_code"])
    .drop_duplicates("project_phase_code")
    .set_index("project_phase_code")["project_phase_label"]
    .to_dict()
)

phase_display_by_code = {
    str(code): (
        f"{label} ({code})"
        if label and str(label) != str(code)
        else str(code)
    )
    for code, label in phase_display_by_code.items()
}

phase_mapping_df = pd.DataFrame(
    [
        {
            "project_phase_code": code,
            "project_phase_label": (
                phase_display_by_code[code]
                .removesuffix(f" ({code})")
            ),
            "aia_count": int(
                project_phase_data[
                    "project_phase_code"
                ].astype(str).eq(code).sum()
            ),
        }
        for code in sorted(phase_display_by_code)
    ]
)

phase_answered = len(project_phase_data)
phase_json_total = int(assessment_report["has_usable_json"].sum())

display(
    Markdown(
        f"**Project Phase coverage:** {phase_answered:,} of "
        f"{phase_json_total:,} JSON-backed AIAs "
        f"({100 * phase_answered / phase_json_total:.1f}%) "
        "contain an answered `projectDetailsPhase`."
    )
)
display(phase_mapping_df)

In [ ]:
# @title
phase_year_data = project_phase_data[
    project_phase_data["publication_year"].notna()
].copy()

phase_year = pd.crosstab(
    phase_year_data["publication_year"],
    phase_year_data["project_phase_code"].astype(str),
).sort_index()

phase_year = phase_year.rename(
    columns=phase_display_by_code
)

ax = phase_year.plot(
    kind="bar",
    stacked=True,
    figsize=(10, 5),
)
ax.set_title(
    "JSON-backed AIAs by Project Phase and publication year"
)
ax.set_xlabel("Publication year")
ax.set_ylabel("AIA count")
ax.tick_params(axis="x", rotation=0)

for container in ax.containers:
    labels = [
        str(int(v)) if v > 0 else ""
        for v in container.datavalues
    ]
    ax.bar_label(
        container,
        labels=labels,
        label_type="center",
    )

fig = ax.figure
fig.tight_layout()

save_and_show(
    fig,
    "05_project_phase_by_year.png",
    "JSON-backed AIAs by Project Phase and publication year",
)

In [ ]:
# @title
phase_department = pd.crosstab(
    project_phase_data["organization_en"].fillna("Unknown institution"),
    project_phase_data["project_phase_code"].astype(str),
)

phase_department = phase_department.rename(
    columns=phase_display_by_code
)
phase_department = phase_department.loc[
    phase_department.sum(axis=1).sort_values().index
]

ax = phase_department.plot(
    kind="barh",
    stacked=True,
    figsize=(11, max(5, 0.52 * len(phase_department))),
)
ax.set_title(
    "JSON-backed AIAs by Project Phase and department"
)
ax.set_xlabel("AIA count")
ax.set_ylabel("")

for container in ax.containers:
    labels = [
        str(int(v)) if v > 0 else ""
        for v in container.datavalues
    ]
    ax.bar_label(
        container,
        labels=labels,
        label_type="center",
    )

fig = ax.figure
fig.tight_layout()

save_and_show(
    fig,
    "06_project_phase_by_department.png",
    "JSON-backed AIAs by Project Phase and department",
)

# Selected phase-aware data-quality and data-source answers

For each AIA, the report uses the Design or Implementation data-quality fields that match its saved Project Phase. The GBA Plus public-information follow-up resolves to question 6 for v0.x surveys and question 7 for v1.x surveys. In each paired histogram, the two questions form the x-axis groups and Yes/No are the bars within each group. The conditional follow-up denominator includes records where the first answer is Yes, plus any record that explicitly saved a follow-up answer. Standalone answers are grouped first by publication year and then by department. Every data-quality plot shows the full Design and Implementation wording from the latest survey above the plot.


In [ ]:
# @title
selected_question_coverage = (
    question_answers_df.groupby(
        ["question_id", "question_label"],
        as_index=False,
    )
    .agg(
        applicable_aia_count=("applicable", "sum"),
        answered_aia_count=("answered", "sum"),
    )
)

selected_question_coverage["answer_coverage_pct"] = (
    100
    * selected_question_coverage["answered_aia_count"]
    / selected_question_coverage["applicable_aia_count"].replace(
        0, float("nan")
    )
).round(1)

display(Markdown("### Selected-question answer coverage"))
display(selected_question_coverage)

def combined_data_quality_wording(question_ids: list[str]) -> str:
    return "\n".join(
        DATA_QUALITY_FULL_QUESTION_TEXT[question_id]
        for question_id in question_ids
    )

def add_question_text_above_plot(
    fig,
    ax,
    title: str,
    question_text: str,
):
    wrapped_lines = []
    for line in question_text.splitlines():
        wrapped_lines.extend(textwrap.wrap(line, width=115) or [""])
    wrapped = "\n".join(wrapped_lines)
    fig.set_size_inches(max(fig.get_figwidth(), 11), 6.5 + 0.25 * len(wrapped_lines))
    fig.suptitle(title, y=0.99, fontsize=14)
    fig.text(0.5, 0.94, wrapped, ha="center", va="top", fontsize=9)
    plot_top = max(0.65, 0.94 - 0.025 * len(wrapped_lines))
    fig.tight_layout(rect=[0, 0, 1, plot_top])

for pair_number, (first_id, second_id, pair_title) in enumerate(
    PAIRED_QUESTION_GROUPS,
    start=7,
):
    pair_data = question_answers_df[
        question_answers_df["question_id"].isin([first_id, second_id])
        & question_answers_df["answered"]
    ].copy()

    pair_counts = pd.crosstab(
        pair_data["question_id"],
        pair_data["answer_label"],
    ).reindex(
        index=[first_id, second_id],
        columns=["Yes", "No"],
        fill_value=0,
    )
    pair_counts.index = [
        PAIR_PLOT_LABELS[first_id],
        PAIR_PLOT_LABELS[second_id],
    ]

    ax = pair_counts.plot(
        kind="bar",
        stacked=False,
        figsize=(9, 5),
        width=0.78,
    )
    ax.set_xlabel("Question")
    ax.set_ylabel("AIA count")
    ax.tick_params(axis="x", rotation=0)
    ax.legend(
        title="Answer",
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
    )

    for container in ax.containers:
        ax.bar_label(
            container,
            labels=[
                str(int(value)) if value > 0 else ""
                for value in container.datavalues
            ],
            padding=3,
        )

    fig = ax.figure
    question_text = combined_data_quality_wording([first_id, second_id])
    add_question_text_above_plot(fig, ax, pair_title, question_text)
    save_and_show(
        fig,
        f"{pair_number:02d}_{first_id}_{second_id}.png",
        pair_title,
        question_text,
    )


In [ ]:
# @title
STANDALONE_ANSWER_ORDER = {
    "dataQualityPhase10": ["Yes", "No"],
    "aboutDataSource2": [
        "None",
        "Protected A",
        "Protected B",
        "Protected C",
        "Confidential",
        "Secret",
        "Top Secret",
        "Other",
    ],
}

for plot_number, question_id in enumerate(
    STANDALONE_QUESTION_IDS,
    start=11,
):
    question_data = question_answers_df[
        question_answers_df["question_id"].eq(question_id)
        & question_answers_df["answered"]
        & question_answers_df["publication_year"].notna()
    ].copy()

    answer_order = [
        answer
        for answer in STANDALONE_ANSWER_ORDER[question_id]
        if answer in set(question_data["answer_label"])
    ]
    answer_order.extend(
        sorted(set(question_data["answer_label"]) - set(answer_order))
    )

    year_counts = pd.crosstab(
        question_data["publication_year"],
        question_data["answer_label"],
    ).sort_index()
    year_counts = year_counts.reindex(columns=answer_order, fill_value=0)

    ax = year_counts.plot(
        kind="bar",
        stacked=False,
        figsize=(12, 6),
        width=0.82,
    )
    title = f"{SELECTED_QUESTION_LABELS[question_id]} by publication year"
    ax.set_title(title)
    ax.set_xlabel("Publication year")
    ax.set_ylabel("AIA count")
    ax.tick_params(axis="x", rotation=0)
    ax.legend(title="Answer", bbox_to_anchor=(1.02, 1), loc="upper left")

    for container in ax.containers:
        ax.bar_label(
            container,
            labels=[
                str(int(value)) if value > 0 else ""
                for value in container.datavalues
            ],
            padding=2,
            fontsize=8,
        )

    fig = ax.figure
    question_text = DATA_QUALITY_FULL_QUESTION_TEXT.get(question_id)
    if question_text:
        ax.set_title("")
        add_question_text_above_plot(fig, ax, title, question_text)
    else:
        fig.tight_layout()
    save_and_show(
        fig,
        f"{plot_number:02d}_{question_id}_by_year.png",
        title,
        question_text,
    )


In [ ]:
# @title
for plot_number, question_id in enumerate(
    STANDALONE_QUESTION_IDS,
    start=13,
):
    question_data = question_answers_df[
        question_answers_df["question_id"].eq(question_id)
        & question_answers_df["answered"]
    ].copy()
    question_data["organization_en"] = question_data[
        "organization_en"
    ].fillna("Unknown institution")

    answer_order = [
        answer
        for answer in STANDALONE_ANSWER_ORDER[question_id]
        if answer in set(question_data["answer_label"])
    ]
    answer_order.extend(
        sorted(set(question_data["answer_label"]) - set(answer_order))
    )

    department_counts = pd.crosstab(
        question_data["organization_en"],
        question_data["answer_label"],
    ).reindex(columns=answer_order, fill_value=0)
    department_counts = department_counts.loc[
        department_counts.sum(axis=1).sort_values().index
    ]

    ax = department_counts.plot(
        kind="barh",
        stacked=True,
        figsize=(12, max(6, 0.58 * len(department_counts))),
    )
    title = f"{SELECTED_QUESTION_LABELS[question_id]} by department"
    ax.set_title(title)
    ax.set_xlabel("AIA count")
    ax.set_ylabel("")
    ax.legend(title="Answer", bbox_to_anchor=(1.02, 1), loc="upper left")

    for container in ax.containers:
        ax.bar_label(
            container,
            labels=[
                str(int(value)) if value > 0 else ""
                for value in container.datavalues
            ],
            label_type="center",
            fontsize=8,
        )

    fig = ax.figure
    question_text = DATA_QUALITY_FULL_QUESTION_TEXT.get(question_id)
    if question_text:
        ax.set_title("")
        add_question_text_above_plot(fig, ax, title, question_text)
    else:
        fig.tight_layout()
    save_and_show(
        fig,
        f"{plot_number:02d}_{question_id}_by_department.png",
        title,
        question_text,
    )


# Phase-aware consultation answers

The report selects `consultationDesign1`/`consultationDesign3` or `consultationImplementation1`/`consultationImplementation3` from each AIA's saved Project Phase. Yes/No answer counts are broken down by publication year and by department.


In [ ]:
# @title
CONSULTATION_ANSWER_ORDER = ["Yes", "No"]

for plot_number, question_id in enumerate(
    CONSULTATION_QUESTION_IDS,
    start=15,
):
    question_data = question_answers_df[
        question_answers_df["question_id"].eq(question_id)
        & question_answers_df["answered"]
        & question_answers_df["publication_year"].notna()
    ].copy()

    year_counts = pd.crosstab(
        question_data["publication_year"],
        question_data["answer_label"],
    ).sort_index().reindex(columns=CONSULTATION_ANSWER_ORDER, fill_value=0)

    ax = year_counts.plot(
        kind="bar",
        stacked=False,
        figsize=(12, 6),
        width=0.82,
    )
    title = f"{SELECTED_QUESTION_LABELS[question_id]} by publication year"
    ax.set_title(title)
    ax.set_xlabel("Publication year")
    ax.set_ylabel("AIA count")
    ax.tick_params(axis="x", rotation=0)
    ax.legend(title="Answer", bbox_to_anchor=(1.02, 1), loc="upper left")

    for container in ax.containers:
        ax.bar_label(
            container,
            labels=[str(int(value)) if value > 0 else "" for value in container.datavalues],
            padding=2,
            fontsize=8,
        )

    fig = ax.figure
    fig.tight_layout()
    save_and_show(
        fig,
        f"{plot_number:02d}_{question_id}_by_year.png",
        title,
    )


In [ ]:
# @title
for plot_number, question_id in enumerate(
    CONSULTATION_QUESTION_IDS,
    start=17,
):
    question_data = question_answers_df[
        question_answers_df["question_id"].eq(question_id)
        & question_answers_df["answered"]
    ].copy()
    question_data["organization_en"] = question_data[
        "organization_en"
    ].fillna("Unknown institution")

    department_counts = pd.crosstab(
        question_data["organization_en"],
        question_data["answer_label"],
    ).reindex(columns=CONSULTATION_ANSWER_ORDER, fill_value=0)
    department_counts = department_counts.loc[
        department_counts.sum(axis=1).sort_values().index
    ]

    ax = department_counts.plot(
        kind="barh",
        stacked=True,
        figsize=(12, max(6, 0.58 * len(department_counts))),
    )
    title = f"{SELECTED_QUESTION_LABELS[question_id]} by department"
    ax.set_title(title)
    ax.set_xlabel("AIA count")
    ax.set_ylabel("")
    ax.legend(title="Answer", bbox_to_anchor=(1.02, 1), loc="upper left")

    for container in ax.containers:
        ax.bar_label(
            container,
            labels=[str(int(value)) if value > 0 else "" for value in container.datavalues],
            label_type="center",
            fontsize=8,
        )

    fig = ax.figure
    fig.tight_layout()
    save_and_show(
        fig,
        f"{plot_number:02d}_{question_id}_by_department.png",
        title,
    )


# Peer Review resource coverage for impact level 2+ AIAs

For every AIA with a calculated **impact level of 2, 3, or 4**, this section checks the live CKAN resource list for a separately published peer-review artifact.

A package counts as having a peer-review resource only when at least one **resource title, resource description, or resource URL filename** explicitly matches a peer-review term:

- English: `peer review` / `peer-review`
- French: `examen par les pairs`, `évaluation par les pairs`, or `revue par les pairs`

This deliberately does **not** count package-description statements such as “the project underwent a peer review” when no peer-review resource is actually published.

The chart summarizes resource coverage by institution. The package-level table then shows every impact-level-2+ AIA, its status, and any peer-review resource names and URLs that were detected.

In [ ]:
# @title
peer_review_detail_df = assessment_report[
    assessment_report["peer_review_level_2_plus_scope"]
].copy()

peer_review_detail_df = peer_review_detail_df[
    [
        "package_id",
        "package_title_en",
        "organization_en",
        "dataset_url",
        "impact_level",
        "peer_review_status",
        "peer_review_resource_count",
        "peer_review_resource_names",
        "peer_review_resource_urls",
    ]
].sort_values(
    [
        "peer_review_status",
        "organization_en",
        "impact_level",
        "package_title_en",
    ],
    ascending=[True, True, False, True],
)

peer_review_plot_df = (
    assessment_report[
        assessment_report["peer_review_level_2_plus_scope"]
    ]
    .groupby("organization_en", dropna=False)
    .agg(
        peer_review_found=("peer_review_found_in_scope", "sum"),
        peer_review_missing=("peer_review_missing_in_scope", "sum"),
        level_2_plus_aia_count=("package_id", "count"),
    )
    .sort_values(
        ["level_2_plus_aia_count", "peer_review_found"],
        ascending=True,
    )
)

ax = peer_review_plot_df[
    ["peer_review_found", "peer_review_missing"]
].plot(
    kind="barh",
    stacked=True,
    figsize=(11, max(5, 0.6 * len(peer_review_plot_df))),
)

ax.set_title(
    "Published peer-review resource coverage for impact level 2+ AIAs"
)
ax.set_xlabel("Impact level 2+ AIA count")
ax.set_ylabel("")
ax.legend(
    [
        "Peer review resource found",
        "No peer review resource found",
    ],
    loc="lower right",
)

for container in ax.containers:
    labels = [
        str(int(value)) if value > 0 else ""
        for value in container.datavalues
    ]
    ax.bar_label(
        container,
        labels=labels,
        label_type="center",
    )

fig = ax.figure
fig.tight_layout()

save_and_show(
    fig,
    "19_peer_review_resource_coverage_level_2_plus.png",
    "Published peer-review resource coverage for impact level 2+ AIAs",
)

peer_review_summary_df = pd.DataFrame(
    {
        "metric": [
            "Impact level 2+ AIAs checked",
            "Peer review resource found",
            "No peer review resource found",
            "Peer review resource coverage (%)",
        ],
        "value": [
            peer_scope_count,
            peer_found_count,
            peer_missing_count,
            round(peer_coverage_pct, 1)
            if peer_scope_count else None,
        ],
    }
)

display(Markdown("### Peer Review resource coverage"))
display(peer_review_summary_df)

display(Markdown("### Impact level 2+ AIA package check"))
display(peer_review_detail_df)

# Data completeness - all questionnaire questions

This denominator includes every question in the matching version-specific questionnaire schema, including questions that are only shown on conditional branches.

In [ ]:
# @title
institution_summary = (
    assessment_report.groupby(["organization_en", "owner_org"], dropna=False)
    .agg(
        total_aia_count=("package_id", "count"),
        json_backed_aia_count=("has_usable_json", "sum"),
        missing_json_count=("missing_json", "sum"),
        level_2_plus_aia_count=("peer_review_level_2_plus_scope", "sum"),
        peer_review_resource_found_count=("peer_review_found_in_scope", "sum"),
        peer_review_resource_missing_count=("peer_review_missing_in_scope", "sum"),
        avg_completion_all_pct=("completion_all_pct", "mean"),
        median_completion_all_pct=("completion_all_pct", "median"),
        avg_completion_nonconditional_pct=("completion_nonconditional_pct", "mean"),
        median_completion_nonconditional_pct=("completion_nonconditional_pct", "median"),
    )
    .reset_index()
)

institution_summary["json_backed_aia_count"] = (
    institution_summary["json_backed_aia_count"].astype(int)
)
institution_summary["missing_json_count"] = (
    institution_summary["missing_json_count"].astype(int)
)

for column in (
    "level_2_plus_aia_count",
    "peer_review_resource_found_count",
    "peer_review_resource_missing_count",
):
    institution_summary[column] = institution_summary[column].astype(int)

institution_summary["peer_review_resource_coverage_pct"] = (
    100
    * institution_summary["peer_review_resource_found_count"]
    / institution_summary["level_2_plus_aia_count"].replace(0, float("nan")).astype(float)
).round(1)

plot_all = institution_summary.sort_values("avg_completion_all_pct").copy()
plot_all["plot_label"] = plot_all.apply(
    lambda r: (
        f"{r['organization_en']} "
        f"(n={int(r['total_aia_count'])}, "
        f"missing JSON={int(r['missing_json_count'])})"
    ),
    axis=1,
)

fig, ax = plt.subplots(figsize=(12, max(6, 0.58 * len(plot_all))))
bars = ax.barh(plot_all["plot_label"], plot_all["avg_completion_all_pct"])
ax.set_title("Mean AIA question completeness by institution - all questions")
ax.set_xlabel("Mean completeness (%)")
ax.set_ylabel("")
ax.set_xlim(0, 100)
ax.bar_label(
    bars,
    labels=[f"{v:.1f}%" for v in plot_all["avg_completion_all_pct"]],
    padding=3,
)
fig.tight_layout()

save_and_show(
    fig,
    "20_completeness_all_questions_by_institution.png",
    "Mean AIA question completeness by institution - all questions",
)

In [ ]:
# @title
version_all = (
    assessment_report[assessment_report["has_usable_json"]]
    .groupby("aia_version", dropna=False)
    .agg(
        aia_count=("package_id", "count"),
        avg_completion_all_pct=("completion_all_pct", "mean"),
    )
    .reset_index()
    .sort_values("avg_completion_all_pct")
)

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(version_all["aia_version"], version_all["avg_completion_all_pct"])
ax.set_title("Mean question completeness by AIA version - all questions")
ax.set_xlabel("Mean completeness (%)")
ax.set_ylabel("AIA version")
ax.set_xlim(0, 100)
ax.bar_label(
    bars,
    labels=[
        f"{pct:.1f}% (n={int(n)})"
        for pct, n in zip(
            version_all["avg_completion_all_pct"],
            version_all["aia_count"],
        )
    ],
    padding=3,
)
fig.tight_layout()

save_and_show(
    fig,
    "21_completeness_all_questions_by_version.png",
    "Mean question completeness by AIA version - all questions",
)

# Data completeness - non-conditional questions only

This second denominator excludes any question that is conditional itself **or inherits conditionality from a parent page/panel/container**.

In [ ]:
# @title
plot_nonconditional = institution_summary.sort_values(
    "avg_completion_nonconditional_pct"
).copy()

plot_nonconditional["plot_label"] = plot_nonconditional.apply(
    lambda r: (
        f"{r['organization_en']} "
        f"(n={int(r['total_aia_count'])}, "
        f"missing JSON={int(r['missing_json_count'])})"
    ),
    axis=1,
)

fig, ax = plt.subplots(figsize=(12, max(6, 0.58 * len(plot_nonconditional))))
bars = ax.barh(
    plot_nonconditional["plot_label"],
    plot_nonconditional["avg_completion_nonconditional_pct"],
)
ax.set_title(
    "Mean AIA question completeness by institution - non-conditional questions"
)
ax.set_xlabel("Mean completeness (%)")
ax.set_ylabel("")
ax.set_xlim(0, 100)
ax.bar_label(
    bars,
    labels=[
        f"{v:.1f}%"
        for v in plot_nonconditional["avg_completion_nonconditional_pct"]
    ],
    padding=3,
)
fig.tight_layout()

save_and_show(
    fig,
    "22_completeness_nonconditional_by_institution.png",
    "Mean AIA question completeness by institution - non-conditional questions",
)

In [ ]:
# @title
version_nonconditional = (
    assessment_report[assessment_report["has_usable_json"]]
    .groupby("aia_version", dropna=False)
    .agg(
        aia_count=("package_id", "count"),
        avg_completion_nonconditional_pct=("completion_nonconditional_pct", "mean"),
    )
    .reset_index()
    .sort_values("avg_completion_nonconditional_pct")
)

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(
    version_nonconditional["aia_version"],
    version_nonconditional["avg_completion_nonconditional_pct"],
)
ax.set_title(
    "Mean question completeness by AIA version - non-conditional questions"
)
ax.set_xlabel("Mean completeness (%)")
ax.set_ylabel("AIA version")
ax.set_xlim(0, 100)
ax.bar_label(
    bars,
    labels=[
        f"{pct:.1f}% (n={int(n)})"
        for pct, n in zip(
            version_nonconditional["avg_completion_nonconditional_pct"],
            version_nonconditional["aia_count"],
        )
    ],
    padding=3,
)
fig.tight_layout()

save_and_show(
    fig,
    "23_completeness_nonconditional_by_version.png",
    "Mean question completeness by AIA version - non-conditional questions",
)

## 7. Export the three analytical CSVs

The report intentionally keeps the tabular outputs small:

1. `aia_report_assessments.csv` - one row per published AIA, including raw/decoded Project Phase;
2. `aia_report_institutions.csv` - one row per institution, including Project Phase counts; and
3. `aia_report_yearly.csv` - one row per publication year, including Project Phase counts.

In [ ]:
# @title
institution_summary = institution_summary.sort_values(
    ["total_aia_count", "organization_en"],
    ascending=[False, True],
).reset_index(drop=True)

def _output_slug(value: str) -> str:
    slug = re.sub(r"[^a-z0-9]+", "_", str(value).lower()).strip("_")
    return slug or "unknown"

phase_output_source = assessment_report[
    assessment_report["project_phase_code"].notna()
].copy()

if not phase_output_source.empty:
    phase_output_source["phase_column"] = phase_output_source.apply(
        lambda r: (
            "project_phase_"
            + _output_slug(r["project_phase_code"])
            + "_"
            + _output_slug(r["project_phase_label"])
            + "_count"
        ),
        axis=1,
    )

    phase_institution_wide = (
        phase_output_source.pivot_table(
            index=["organization_en", "owner_org"],
            columns="phase_column",
            values="package_id",
            aggfunc="count",
            fill_value=0,
        )
        .reset_index()
    )

    institution_summary = institution_summary.merge(
        phase_institution_wide,
        on=["organization_en", "owner_org"],
        how="left",
    )

    for column in institution_summary.columns:
        if column.startswith("project_phase_") and column.endswith("_count"):
            institution_summary[column] = (
                institution_summary[column].fillna(0).astype(int)
            )

base_yearly = (
    assessment_report.dropna(subset=["publication_year"])
    .groupby("publication_year")
    .agg(
        total_aia_count=("package_id", "count"),
        json_backed_aia_count=("has_usable_json", "sum"),
        missing_json_count=("missing_json", "sum"),
        level_2_plus_aia_count=("peer_review_level_2_plus_scope", "sum"),
        peer_review_resource_found_count=("peer_review_found_in_scope", "sum"),
        peer_review_resource_missing_count=("peer_review_missing_in_scope", "sum"),
        mean_completion_all_pct=("completion_all_pct", "mean"),
        mean_completion_nonconditional_pct=("completion_nonconditional_pct", "mean"),
    )
)

impact_source = assessment_report[
    assessment_report["publication_year"].notna()
    & assessment_report["impact_level"].notna()
].copy()

impact_source["impact_column"] = (
    impact_source["impact_level"]
    .astype(int)
    .map(lambda x: f"impact_level_{x}_count")
)

impact_wide = impact_source.pivot_table(
    index="publication_year",
    columns="impact_column",
    values="package_id",
    aggfunc="count",
    fill_value=0,
)

version_source = assessment_report[
    assessment_report["publication_year"].notna()
    & assessment_report["aia_version"].notna()
].copy()

version_source["version_column"] = (
    "version_"
    + version_source["aia_version"]
    .astype(str)
    .str.replace(".", "_", regex=False)
    .str.replace("-", "_", regex=False)
    + "_count"
)

version_wide = version_source.pivot_table(
    index="publication_year",
    columns="version_column",
    values="package_id",
    aggfunc="count",
    fill_value=0,
)

phase_year_wide = pd.DataFrame(index=base_yearly.index)

if not phase_output_source.empty:
    phase_year_source = phase_output_source[
        phase_output_source["publication_year"].notna()
    ].copy()

    phase_year_wide = phase_year_source.pivot_table(
        index="publication_year",
        columns="phase_column",
        values="package_id",
        aggfunc="count",
        fill_value=0,
    )

yearly_summary = (
    base_yearly.join(impact_wide, how="left")
    .join(version_wide, how="left")
    .join(phase_year_wide, how="left")
    .fillna(0)
    .reset_index()
    .sort_values("publication_year")
)

for column in yearly_summary.columns:
    if column.endswith("_count"):
        yearly_summary[column] = yearly_summary[column].astype(int)

yearly_summary["peer_review_resource_coverage_pct"] = (
    100
    * yearly_summary["peer_review_resource_found_count"]
    / yearly_summary["level_2_plus_aia_count"].replace(0, float("nan")).astype(float)
).round(1)

assessment_csv = OUTPUT_DIR / "aia_report_assessments.csv"
institution_csv = OUTPUT_DIR / "aia_report_institutions.csv"
yearly_csv = OUTPUT_DIR / "aia_report_yearly.csv"

assessment_report.to_csv(assessment_csv, index=False)
institution_summary.to_csv(institution_csv, index=False)
yearly_summary.to_csv(yearly_csv, index=False)

display(Markdown("### Institution summary"))
display(institution_summary)

display(Markdown("### Yearly summary"))
display(yearly_summary)

print("CSV outputs:")
print(" -", assessment_csv)
print(" -", institution_csv)
print(" -", yearly_csv)

## 8. Generate a standalone HTML report

This cell builds the HTML directly from the **current in-memory run**. No previously rendered notebook or stored report is needed. The ten PNG figures are embedded as base64 data URIs so the HTML file is portable.

In [ ]:
# @title
def image_data_uri(path: Path) -> str:
    encoded = base64.b64encode(path.read_bytes()).decode("ascii")
    return f"data:image/png;base64,{encoded}"

def table_html(df: pd.DataFrame, *, max_rows: int = 100) -> str:
    return df.head(max_rows).to_html(
        index=False,
        border=0,
        classes="data-table",
        escape=True,
    )

generated_at = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC")

plot_sections = "\n".join(
    f'''
    <section class="figure">
      <h3>{html.escape(title)}</h3>
      {
        f'<p class="question-text">{html.escape(question_text).replace(chr(10), "<br>")}</p>'
        if question_text else ""
      }
      <img src="{image_data_uri(path)}"
           alt="{html.escape(title)}">
    </section>
    '''
    for title, path, question_text in PLOT_FILES
)

institution_html = table_html(institution_summary)
yearly_html = table_html(yearly_summary)

report_html = f'''<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Government of Canada AIA analysis report</title>
<style>
  :root {{ color-scheme: light dark; }}
  body {{
    font-family: system-ui, -apple-system, BlinkMacSystemFont,
                 "Segoe UI", sans-serif;
    line-height: 1.55;
    max-width: 1200px;
    margin: 0 auto;
    padding: 2rem;
  }}
  h1, h2, h3 {{ line-height: 1.2; }}
  .lede {{ font-size: 1.1rem; }}
  .metrics {{
    display: grid;
    grid-template-columns: repeat(auto-fit, minmax(180px, 1fr));
    gap: 1rem;
    margin: 1.5rem 0 2rem;
  }}
  .metric {{
    border: 1px solid #8886;
    border-radius: .5rem;
    padding: 1rem;
  }}
  .metric strong {{
    display: block;
    font-size: 1.7rem;
  }}
  .figure {{ margin: 2.5rem 0; }}
  .figure img {{ width: 100%; height: auto; }}
  .question-text {{ font-size: .95rem; margin-bottom: 1rem; }}
  .data-table {{
    border-collapse: collapse;
    width: 100%;
    font-size: .9rem;
  }}
  .data-table th,
  .data-table td {{
    border-bottom: 1px solid #8885;
    padding: .45rem;
    text-align: left;
    vertical-align: top;
  }}
  .note {{
    border-left: .3rem solid #777;
    padding-left: 1rem;
  }}
</style>
</head>
<body>

<h1>Government of Canada Algorithmic Impact Assessments</h1>
<p class="lede">
Live publication, impact-level, questionnaire-version,
and structural data-completeness analysis.
</p>

<p>
Generated {html.escape(generated_at)} from the Open Government Portal
CKAN <code>collection:aia</code> catalogue and the published AIA JSON
resources.
</p>

<div class="metrics">
  <div class="metric"><span>AIA records</span><strong>{total:,}</strong></div>
  <div class="metric"><span>Institutions</span><strong>{institution_count:,}</strong></div>
  <div class="metric"><span>Usable JSON</span><strong>{json_count:,}</strong></div>
  <div class="metric"><span>Missing JSON</span><strong>{missing_count:,}</strong></div>
  <div class="metric">
    <span>Mean completeness</span>
    <strong>{overall_all:.1f}%</strong>
    <span>all questions</span>
  </div>
  <div class="metric">
    <span>Mean completeness</span>
    <strong>{overall_nonconditional:.1f}%</strong>
    <span>non-conditional only</span>
  </div>
</div>

<h2>Figures</h2>
{plot_sections}

<h2>Institution summary</h2>
{institution_html}

<h2>Yearly summary</h2>
{yearly_html}

<h2>Method notes</h2>
<ul>
  <li>
    CKAN package <code>5423054a-093c-4239-85be-fa0b36ae0b2e</code>
    is excluded because it is AIA documentation rather than an AIA analysis.
  </li>
  <li>AIAs without usable JSON are retained and score 0% completeness.</li>
  <li>Impact level is derived only for successfully parsed JSON-backed AIAs.</li>
  <li>
    Peer-review coverage is checked only for impact level 2+ AIAs and
    requires a separately published CKAN resource whose title,
    description, or URL explicitly identifies it as a peer review.
    A peer-review mention in the package description alone does not count.
  </li>
  <li>
    Project Phase is read from <code>projectDetailsPhase</code> and decoded
    with the questionnaire definition matching each AIA version.
  </li>
  <li>
    Selected data-quality and consultation answers use the Design or
    Implementation fields matching each AIA's saved Project Phase. The GBA Plus
    public-information field resolves to question 6 for v0.x and question 7
    for v1.x. Answer codes are decoded with the matching questionnaire.
    Conditional public-availability follow-ups are applicable when their first
    question is Yes, or when an explicit follow-up answer is saved.
  </li>
  <li>
    Non-conditional completeness excludes questions with their own
    <code>visibleIf</code> and questions nested under conditional containers.
  </li>
  <li>
    Completeness measures structural population of questionnaire fields,
    not accuracy, quality, validity, or policy compliance.
  </li>
</ul>

<p class="note">
The notebook that generated this report requires no stored analytical
input files. It begins with the public Open Government Portal CKAN API,
then downloads the current published JSON resources and AIA questionnaire
schemas.
</p>

</body>
</html>
'''

html_path = OUTPUT_DIR / "aia_analysis_report.html"
html_path.write_text(report_html, encoding="utf-8")

print("HTML report:", html_path)

## 9. Final audit

These checks catch accidental exclusion of catalogue records, duplicate AIA rows, invalid impact-level values, or missing output files.

In [ ]:
# @title
checks = {
    "one_row_per_package":
        assessment_report["package_id"].is_unique,

    "all_analytical_packages_preserved":
        len(assessment_report) == len(packages_df),

    "excluded_documentation_record_absent":
        assessment_report["package_id"].isin(EXCLUDED_PACKAGE_IDS).sum() == 0,

    "project_phase_codes_decode_when_present":
        assessment_report.loc[
            assessment_report["project_phase_code"].notna(),
            "project_phase_label",
        ].notna().all(),

    "selected_question_rows_are_unique":
        not question_answers_df.duplicated(
            ["package_id", "question_id"]
        ).any(),

    "selected_question_output_columns_exported":
        set(SELECTED_QUESTION_OUTPUT_COLUMNS).issubset(
            assessment_report.columns
        ),

    "answered_selected_questions_are_applicable":
        question_answers_df.loc[
            question_answers_df["answered"],
            "applicable",
        ].all(),

    "phase_question_sources_match_project_phase":
        question_answers_df.loc[
            question_answers_df["question_id"].str.startswith(
                ("dataQualityPhase", "consultationPhase")
            )
            & question_answers_df["project_phase_label"].isin(
                ["Design", "Implementation"]
            ),
            ["question_id", "project_phase_label", "source_question_id"],
        ].apply(
            lambda r: r["source_question_id"].startswith(
                ("dataQuality" if r["question_id"].startswith("dataQuality") else "consultation")
                + r["project_phase_label"]
            ),
            axis=1,
        ).all(),

    "gba_public_question_matches_version":
        question_answers_df.loc[
            question_answers_df["question_id"].eq(
                "dataQualityPhaseGbaPublic"
            ),
            ["aia_version", "source_question_id"],
        ].apply(
            lambda r: r["source_question_id"].endswith(
                "7" if str(r["aia_version"]).lower().lstrip("v").startswith("1.") else "6"
            ),
            axis=1,
        ).all(),

    "missing_json_scored_zero_all":
        assessment_report.loc[
            assessment_report["missing_json"],
            "completion_all_pct",
        ].eq(0).all(),

    "missing_json_scored_zero_nonconditional":
        assessment_report.loc[
            assessment_report["missing_json"],
            "completion_nonconditional_pct",
        ].eq(0).all(),

    "impact_levels_are_1_to_4":
        assessment_report["impact_level"].dropna().between(1, 4).all(),

    "peer_review_scope_is_level_2_plus_only":
        assessment_report.loc[
            assessment_report["peer_review_level_2_plus_scope"],
            "impact_level",
        ].ge(2).all(),

    "peer_review_status_complete_for_level_2_plus":
        assessment_report.loc[
            assessment_report["peer_review_level_2_plus_scope"],
            "peer_review_status",
        ].isin(
            [
                "Peer review resource found",
                "No peer review resource found",
            ]
        ).all(),

    "three_csv_outputs_exist":
        all(
            path.exists()
            for path in (assessment_csv, institution_csv, yearly_csv)
        ),

    "html_output_exists":
        html_path.exists(),

    "twenty_three_plots_generated":
        len(PLOT_FILES) == 23,
}

checks_df = pd.DataFrame(
    checks.items(),
    columns=["check", "passed"],
)

display(checks_df)

if not checks_df["passed"].all():
    raise AssertionError("One or more report integrity checks failed.")

print("All report integrity checks passed.")